In [1]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, chi2

file_path = 'train_ver2.csv'
last_date = '2016-05-28'

header = pd.read_csv(file_path, nrows=0).columns

product_cols = [
    col for col in header
    if col.startswith('ind_') and col.endswith('_ult1')
]

cols_q1 = ['fecha_dato', 'nomprov', 'renta', 'age', 'ncodpers']

df_q1 = pd.read_csv(
    file_path,
    usecols=cols_q1,
    dtype={'age': float, 'renta': float},
    skipinitialspace=True,
    na_values=[' NA']
)

df_q1 = df_q1.loc[df_q1['fecha_dato'] == last_date].copy()
df_q1 = df_q1.dropna(subset=['nomprov', 'ncodpers'])

grouped = df_q1.groupby('nomprov')

q1_result = pd.DataFrame({
    'customer_count': grouped['ncodpers'].count(),
    'mean_income': grouped['renta'].mean(),
    'median_income': grouped['renta'].median(),
    'mean_age': grouped['age'].mean()
}).reset_index()

q1_result = q1_result.loc[q1_result['customer_count'] >= 50]

q1_top5 = q1_result.sort_values(
    by=['mean_income', 'nomprov'],
    ascending=[False, True]
).head(5).reset_index(drop=True)

print("=== Q1. 평균 소득 상위 5개 도시 ===")
q1_top5

=== Q1. 평균 소득 상위 5개 도시 ===


,nomprov,customer_count,mean_income,median_income,mean_age
0,CEUTA,513,200515.885152,128699.580,44.007797
1,MADRID,298841,178362.993637,138024.135,45.205564
2,"BALEARS, ILLES",9143,171890.951564,122837.220,39.748770
3,BARCELONA,88754,164584.843157,130877.160,39.865144
4,MELILLA,641,149071.944149,117718.365,45.329173


In [2]:
cols_q2 = ['fecha_dato', 'ncodpers'] + product_cols

df_q2 = pd.read_csv(
    file_path,
    usecols=cols_q2,
    dtype={col: 'float32' for col in product_cols},
    skipinitialspace=True,
    na_values=[' NA']
)

df_q2 = df_q2.dropna(subset=['fecha_dato', 'ncodpers'])

monthly_rows = []
previous = None
previous_month = None

for date, current in df_q2.groupby('fecha_dato'):
    month_number = int(date[:4]) * 12 + int(date[5:7])
    counts = {col: 0 for col in product_cols}

    if previous is not None and month_number - previous_month == 1:
        joined = current.join(
            previous.set_index('ncodpers'),
            on='ncodpers'
        )

        for col in product_cols:
            is_new = (
                (joined[col] == 1)
                & (joined[col + '_prev'] == 0)
            )
            counts[col] = is_new.sum()

    counts['fecha_dato'] = date
    monthly_rows.append(counts)

    previous = current[['ncodpers'] + product_cols].rename(
        columns={col: col + '_prev' for col in product_cols}
    )
    previous_month = month_number

q2_monthly = pd.DataFrame(monthly_rows).set_index('fecha_dato')

q2_result = pd.DataFrame({
    'product': product_cols,
    'new_purchases': [
        q2_monthly[col].sum() for col in product_cols
    ]
})

q2_top5 = q2_result.sort_values(
    by=['new_purchases', 'product'],
    ascending=[False, True]
).head(5).reset_index(drop=True)

print("=== Q2. 전체 기간 신규 가입 상위 5개 상품 ===")
q2_top5

=== Q2. 전체 기간 신규 가입 상위 5개 상품 ===


,product,new_purchases
0,ind_recibo_ult1,153146
1,ind_nom_pens_ult1,84709
2,ind_nomina_ult1,73753
3,ind_tjcr_fin_ult1,69118
4,ind_cco_fin_ult1,69103


In [5]:
import warnings
warnings.filterwarnings('ignore')

q3_features = [
    'sexo',
    'age',
    'segmento',
    'ind_empleado',
    'tiprel_1mes'
]

cols_q3 = ['fecha_dato'] + q3_features + product_cols

df_q3 = pd.read_csv(
    file_path,
    usecols=cols_q3,
    dtype={'age': float},
    skipinitialspace=True,
    na_values=[' NA'],
    low_memory=False
)

df_q3 = df_q3.loc[df_q3['fecha_dato'] == last_date].copy()

n_before = len(df_q3)
df_q3 = df_q3.dropna().copy()

df_q3['total_products'] = df_q3[product_cols].sum(axis=1)
y = df_q3['total_products'] >= 3

df_q3['age_group'] = 'adult'
df_q3.loc[df_q3['age'] < 30, 'age_group'] = 'young'
df_q3.loc[df_q3['age'] >= 65, 'age_group'] = 'senior'

target_features = {
    '성별 (sexo)': 'sexo',
    '나이 구간 (age_group)': 'age_group',
    '세그먼트 (segmento)': 'segmento',
    '직원 구분 (ind_empleado)': 'ind_empleado',
    '고객 관계 유형 (tiprel_1mes)': 'tiprel_1mes'
}

results = []

for name, col in target_features.items():
    X = pd.get_dummies(df_q3[col])

    model = SelectKBest(
        score_func=chi2,
        k=len(X.columns)
    )
    model.fit(X, y)

    results.append({
        '기준': name,
        'Chi-Square Score': model.scores_.sum()
    })

q3_result = pd.DataFrame(results).sort_values(
    by='Chi-Square Score',
    ascending=False
).reset_index(drop=True)

print("=== Q3. 기준별 카이제곱 통계량 ===")
print(f"분석 대상 고객 수: {len(df_q3):,}명")
print(f"결측값으로 제외된 고객 수: {n_before - len(df_q3):,}명")
print(
    "\n카이제곱 통계량 기준으로 가장 관련성이 큰 기준:",
    q3_result.iloc[0]['기준']
)

q3_result

=== Q3. 기준별 카이제곱 통계량 ===
분석 대상 고객 수: 924,410명
결측값으로 제외된 고객 수: 7,043명

카이제곱 통계량 기준으로 가장 관련성이 큰 기준: 고객 관계 유형 (tiprel_1mes)


,기준,Chi-Square Score
0,고객 관계 유형 (tiprel_1mes),184979.998993
1,세그먼트 (segmento),124374.223984
2,나이 구간 (age_group),61536.478399
3,성별 (sexo),3443.992615
4,직원 구분 (ind_empleado),1336.294946
